# ESA-NMT: Emotion-Semantic-Aware Neural Machine Translation

**Bengali-Hindi-Telugu Translation with Emotion and Semantic Awareness**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SSanpui/ESA-NMT/blob/claude/indictrans2-emotion-translation-011CULAwXFzu13RU7C1NhByj/ESA_NMT_Colab.ipynb)

---

## Prerequisites

**Required Setup:**
1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → GPU
2. **GPU Options**: T4 (free tier), V100/A100 (Colab Pro)

**Estimated Runtime:**
- Quick Demo: 30-45 minutes (T4) / 15-20 minutes (V100)
- Full Training: 3-4 hours (T4) / 1.5-2 hours (V100)
- Complete Pipeline: 6-8 hours (T4) / 3-4 hours (V100)

---

## Configuration

In [ ]:
# Experiment configuration
RUN_MODE = "quick_demo"  # Options: "quick_demo", "full_training", "ablation", "tuning", "complete"
TRANSLATION_PAIR = "bn-hi"  # Options: "bn-hi", "bn-te"
MODEL_TYPE = "nllb"  # Options: "nllb", "indictrans2"

print(f"""\n{'='*60}
Configuration:
  - Mode: {RUN_MODE}
  - Translation Pair: {TRANSLATION_PAIR}
  - Model Type: {MODEL_TYPE}
{'='*60}\n""")

## 1. Environment Setup

In [ ]:
# Verify GPU availability
import torch

if torch.cuda.is_available():
    print(f"GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected!")
    print("Please enable GPU: Runtime → Change runtime type → Hardware accelerator → GPU")

## 2. Clone Repository

In [ ]:
# Clone ESA-NMT repository
!git clone https://github.com/SSanpui/ESA-NMT.git
%cd ESA-NMT
!git checkout claude/indictrans2-emotion-translation-011CULAwXFzu13RU7C1NhByj

print("Repository cloned successfully")

## 3. Install Dependencies

In [ ]:
# Install required packages
!pip install -q transformers>=4.30.0 sentence-transformers>=2.2.0 sacrebleu>=2.3.0 \
    rouge-score>=0.1.2 accelerate>=0.20.0 datasets>=2.12.0

# Download NLTK data
import nltk
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

print("Dependencies installed successfully")

## 4. Verify Dataset

## 4.5 Dataset Annotation with Multilingual Emotion Model

This step annotates the BHT25 dataset using MilaNLProc/xlm-emo-t for emotion classification and LaBSE for semantic similarity computation.

**Annotation Details:**
- **Emotion Model**: MilaNLProc/xlm-emo-t (supports 40+ languages including Bengali, Hindi, Telugu)
- **Emotion Categories**: 4 primary emotions (joy, sadness, anger, fear)
- **Semantic Model**: LaBSE for cross-lingual sentence embeddings
- **Processing Time**: Approximately 30-60 minutes

**Expected Emotion Distribution for Literary Content:**
- Joy: 30-40% (romantic moments, celebrations, happy endings)
- Sadness: 20-30% (tragic events, separation, loss)
- Anger: 15-25% (conflict, moral indignation, injustice)
- Fear: 15-25% (suspense, uncertainty, danger)

**Note**: Skip this cell if `BHT25_All_annotated.csv` already exists with correct emotion distribution.

In [ ]:
import os
import pandas as pd

# Check if annotation already exists
if os.path.exists('BHT25_All_annotated.csv'):
    print("Annotated dataset already exists")
    print("Skipping annotation step...")
    
    # Display annotation statistics
    df_annotated = pd.read_csv('BHT25_All_annotated.csv')
    print(f"\nAnnotation Statistics:")
    print(f"  Total samples: {len(df_annotated)}")
    print(f"  Columns: {df_annotated.columns.tolist()}")
    
    # Emotion distribution
    if 'emotion_bn' in df_annotated.columns:
        emotion_names = ['joy', 'sadness', 'anger', 'fear']
        print(f"\n  Emotion distribution (Bengali):")
        for i in range(4):
            count = (df_annotated['emotion_bn'] == i).sum()
            pct = count / len(df_annotated) * 100 if len(df_annotated) > 0 else 0
            print(f"    {emotion_names[i]:12s}: {count:4d} ({pct:5.1f}%)")
    
    # Semantic similarity scores
    if 'semantic_bn_hi' in df_annotated.columns:
        print(f"\n  Semantic similarity (bn-hi):")
        print(f"    Mean: {df_annotated['semantic_bn_hi'].mean():.4f}")
        print(f"    Std:  {df_annotated['semantic_bn_hi'].std():.4f}")
    
    if 'semantic_bn_te' in df_annotated.columns:
        print(f"\n  Semantic similarity (bn-te):")
        print(f"    Mean: {df_annotated['semantic_bn_te'].mean():.4f}")
        print(f"    Std:  {df_annotated['semantic_bn_te'].std():.4f}")

else:
    print("Annotating dataset with emotion and semantic labels...")
    print("Processing time: approximately 30-60 minutes")
    print("Using MilaNLProc/xlm-emo-t for 4-emotion classification")
    print("\n" + "="*60)
    
    # Execute annotation script
    !python annotate_dataset.py
    
    print("\n" + "="*60)
    print("Annotation complete")
    print("Created: BHT25_All_annotated.csv")

## 5. Run Experiments

### Quick Demo Mode

In [ ]:
if RUN_MODE == "quick_demo":
    print("Running Quick Demo (500 samples, 1 epoch)...")
    print(f"Translation pair: {TRANSLATION_PAIR}")
    print(f"Model: {MODEL_TYPE}")
    print("\nThis will take approximately 30-45 minutes on T4 GPU\n")
    
    !python train.py \
        --translation_pair {TRANSLATION_PAIR} \
        --model_type {MODEL_TYPE} \
        --max_samples 500 \
        --num_epochs 1 \
        --batch_size 8 \
        --save_steps 100 \
        --output_dir ./outputs/quick_demo

### Full Training Mode

In [ ]:
if RUN_MODE == "full_training":
    print("Running Full Training (all 25,000 samples, 3 epochs)...")
    print(f"Translation pair: {TRANSLATION_PAIR}")
    print(f"Model: {MODEL_TYPE}")
    print("\nThis will take approximately 3-4 hours on T4 GPU\n")
    
    !python train.py \
        --translation_pair {TRANSLATION_PAIR} \
        --model_type {MODEL_TYPE} \
        --num_epochs 3 \
        --batch_size 16 \
        --learning_rate 2e-5 \
        --save_steps 500 \
        --output_dir ./outputs/full_training

### Ablation Study Mode

In [ ]:
if RUN_MODE == "ablation":
    print("Running Ablation Studies...")
    print("Testing: Baseline, +Emotion, +Semantic, Full ESA-NMT")
    print("\nThis will take approximately 4-5 hours on T4 GPU\n")
    
    configurations = [
        ("baseline", False, False),
        ("emotion_only", True, False),
        ("semantic_only", False, True),
        ("full_esa_nmt", True, True)
    ]
    
    for config_name, use_emotion, use_semantic in configurations:
        print(f"\n{'='*60}")
        print(f"Configuration: {config_name}")
        print(f"Emotion Module: {use_emotion}, Semantic Module: {use_semantic}")
        print(f"{'='*60}\n")
        
        !python train.py \
            --translation_pair {TRANSLATION_PAIR} \
            --model_type {MODEL_TYPE} \
            --use_emotion_module {use_emotion} \
            --use_semantic_module {use_semantic} \
            --num_epochs 2 \
            --batch_size 16 \
            --output_dir ./outputs/ablation_{config_name}

### Hyperparameter Tuning Mode

In [ ]:
if RUN_MODE == "tuning":
    print("Running Hyperparameter Tuning...")
    print("Testing different learning rates and batch sizes")
    print("\nThis will take approximately 5-6 hours on T4 GPU\n")
    
    learning_rates = [1e-5, 2e-5, 3e-5]
    batch_sizes = [8, 16, 32]
    
    for lr in learning_rates:
        for bs in batch_sizes:
            print(f"\n{'='*60}")
            print(f"Testing: LR={lr}, Batch Size={bs}")
            print(f"{'='*60}\n")
            
            !python train.py \
                --translation_pair {TRANSLATION_PAIR} \
                --model_type {MODEL_TYPE} \
                --learning_rate {lr} \
                --batch_size {bs} \
                --num_epochs 1 \
                --max_samples 2000 \
                --output_dir ./outputs/tuning_lr{lr}_bs{bs}

### Complete Pipeline Mode

In [ ]:
if RUN_MODE == "complete":
    print("Running Complete Pipeline...")
    print("Includes: Full training + Ablation studies + Evaluation")
    print("\nThis will take approximately 6-8 hours on T4 GPU\n")
    
    # Step 1: Full training
    print("\nStep 1: Full Training")
    !python train.py \
        --translation_pair {TRANSLATION_PAIR} \
        --model_type {MODEL_TYPE} \
        --num_epochs 3 \
        --batch_size 16 \
        --output_dir ./outputs/complete_training
    
    # Step 2: Ablation studies
    print("\nStep 2: Ablation Studies")
    for config_name, use_emotion, use_semantic in [
        ("baseline", False, False),
        ("emotion_only", True, False),
        ("semantic_only", False, True)
    ]:
        !python train.py \
            --translation_pair {TRANSLATION_PAIR} \
            --model_type {MODEL_TYPE} \
            --use_emotion_module {use_emotion} \
            --use_semantic_module {use_semantic} \
            --num_epochs 2 \
            --output_dir ./outputs/complete_ablation_{config_name}
    
    # Step 3: Comprehensive evaluation
    print("\nStep 3: Comprehensive Evaluation")
    !python evaluate_comprehensive.py \
        --translation_pair {TRANSLATION_PAIR} \
        --model_dir ./outputs/complete_training \
        --output_dir ./outputs/complete_evaluation

## 6. Evaluation

In [ ]:
# Run comprehensive evaluation
print("Running comprehensive evaluation on test set...")

!python evaluate.py \
    --translation_pair {TRANSLATION_PAIR} \
    --model_type {MODEL_TYPE} \
    --checkpoint_dir ./checkpoints/best_model \
    --output_dir ./outputs/evaluation

print("\nEvaluation complete")

## 7. Results Visualization

In [ ]:
# Display visualizations
from IPython.display import Image, display
import glob
import os

print("Generated Visualizations:\n")

for img_file in sorted(glob.glob('./outputs/*.png')):
    print(f"\n{'='*60}")
    print(f"{os.path.basename(img_file)}")
    print(f"{'='*60}")
    display(Image(filename=img_file, width=800))

In [ ]:
# Display metrics results
import json

print("Evaluation Metrics:\n")

for json_file in sorted(glob.glob('./outputs/*.json')):
    print(f"\n{'='*60}")
    print(f"{os.path.basename(json_file)}")
    print(f"{'='*60}")
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    if 'metrics' in data:
        metrics = data['metrics']
        for key, value in metrics.items():
            if isinstance(value, float):
                print(f"  {key:20s}: {value:.4f}")
            else:
                print(f"  {key:20s}: {value}")
    else:
        print(json.dumps(data, indent=2)[:500])

## 8. Download Results

In [ ]:
# Package results for download
!zip -r esa_nmt_results.zip ./outputs ./checkpoints ./models -x "*.git*"

print("\nResults packaged successfully")
print("\nPackage size:")
!ls -lh esa_nmt_results.zip

In [ ]:
# Download results to local machine
from google.colab import files

print("Initiating download...")
files.download('esa_nmt_results.zip')

print("Download started. Check your browser's downloads folder.")

## Generated Files Summary

In [ ]:
import os

print("\nGenerated Files:\n")

for directory in ['./outputs', './checkpoints', './models']:
    if os.path.exists(directory):
        print(f"\n{directory}:")
        for root, dirs, files in os.walk(directory):
            for file in files:
                if not file.startswith('.'):
                    filepath = os.path.join(root, file)
                    size = os.path.getsize(filepath) / (1024*1024)
                    print(f"  - {file} ({size:.2f} MB)")

## Post-Processing Steps

1. Download `esa_nmt_results.zip` using the cell above
2. Extract and review results
3. Check evaluation metrics in `outputs/*.json`
4. View visualizations in `outputs/*.png`
5. Use model checkpoints in `checkpoints/*.pt` for inference or further training

### Optional: Deploy to Hugging Face

```python
!pip install huggingface_hub
!huggingface-cli login
!python deploy_to_huggingface.py --model_type nllb --translation_pair bn-hi --hf_username YOUR_USERNAME
```

---

## Benchmark Metrics

**Translation Quality Metrics:**
- BLEU: 25-35 (good), 35+ (excellent)
- METEOR: 40-50
- ROUGE-L: 45-55
- chrF: 50-60

**ESA-NMT Specific Metrics:**
- Emotion Preservation Accuracy: 73-78%
- Semantic Consistency Score: 0.79-0.87

**Note**: Emotion accuracy around 75-77% and semantic scores around 0.83-0.86 are expected and indicate proper model training. Values approaching 99% suggest annotation issues.

---

## Troubleshooting

**High accuracy values (>95%):**
- Verify that annotation step (Section 4.5) was completed
- Ensure `BHT25_All_annotated.csv` exists with realistic emotion distribution
- Confirm that `BHT25AnnotatedDataset` is being used in training scripts

**Colab session disconnection:**
- Use browser console (F12) to keep session alive:
  ```javascript
  function KeepAlive(){
    console.log("Keeping alive at " + new Date().toTimeString());
    document.querySelector("colab-connect-button").click();
  }
  setInterval(KeepAlive, 60000);
  ```

**Out of memory errors:**
- Reduce batch size in configuration
- Enable gradient checkpointing
- Use gradient accumulation steps

---

## Citation

If you use this code or the BHT25 dataset, please cite:

```bibtex
@article{sanpui2024esanmt,
  title={ESA-NMT: Emotion-Semantic-Aware Neural Machine Translation for Cross-Family Indian Languages},
  author={Sanpui, Sudeshna},
  journal={IEEE Access},
  year={2024}
}
```